In [1]:
import tensorflow as tf
import numpy as np
# import tensorflow_datasets as tfds
from datasets import load_dataset, load_from_disk

In [2]:
import os
try:
    import tensorflow_datasets as tfds
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow_datasets'])
    import tensorflow_datasets as tfds

In [3]:
tf.keras.utils.set_random_seed(42)
tf.__version__
tf.executing_eagerly()

True

In [4]:
tiny_texts = [
    "great movie",
    "amazing story",
    "bad movie",
    "terrible story"
]
tiny_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=20,
    output_mode='int',
    output_sequence_length=4,
)
tiny_vectorizer.adapt(tiny_texts)
tiny_vocab = tiny_vectorizer.get_vocabulary()

print("Tiny vocab:")
for index, token in enumerate(tiny_vocab):
    print(index, repr(token))

Tiny vocab:
0 ''
1 '[UNK]'
2 np.str_('story')
3 np.str_('movie')
4 np.str_('terrible')
5 np.str_('great')
6 np.str_('bad')
7 np.str_('amazing')


In [5]:
tiny_sentence = tf.constant(["great movie"])
tiny_ids = tiny_vectorizer(tiny_sentence)

In [6]:
tiny_sentence.numpy()[0].decode("utf-8")

'great movie'

In [7]:
tiny_ids.numpy()

array([[5, 3, 0, 0]])

In [8]:
tiny_ids.shape

TensorShape([1, 4])

### Step 6 -- Turn Token IDs Into Embedding Vectors

Now we can start creating the tiny embedding layer

In [9]:
tiny_embedding = tf.keras.layers.Embedding(
    input_dim=len(tiny_vocab),
    output_dim=3,
)
tiny_vectors = tiny_embedding(tiny_ids)

In [10]:
tiny_vectors.numpy()

array([[[ 0.0265666 , -0.04811701, -0.03076191],
        [ 0.01307596, -0.04135817, -0.00536425],
        [ 0.03581891,  0.01626286, -0.01412892],
        [ 0.03581891,  0.01626286, -0.01412892]]], dtype=float32)

In [11]:
tiny_vectors.shape

TensorShape([1, 4, 3])

### Step 7 -- Understand the Embedding Shape

###### The embedding output shape has three parts:
###### [batch_size, sequence_length, embedding_dim]
###### Each dimension tells a story about how the text is organized for the model

### Part2 - Load a Real Movie Review Dataset

In [12]:
train_data, test_data = tfds.load(
    name='imdb_reviews',
    split=['train[:10000]', 'test[:5000]'],
    as_supervised=True,
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

NonMatchingChecksumError: Artifact http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz, downloaded to C:\Users\OMID\tensorflow_datasets\downloads\imdb_reviews\ai.stanfor.edu_amaas_sentime_aclImdb_v1PaujRp-TxjBWz59jHXsMDm5WiexbxzaFQkEnXc3Tvo8.tar.gz.tmp.ffd9938e3695450c95d9d33a58f1606a\aclImdb_v1.tar.gz, has wrong checksum:
* Expected: UrlInfo(size=80.23 MiB, checksum='c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe', filename='aclImdb_v1.tar.gz')
* Got: UrlInfo(size=391 bytes, checksum='877228606a019cf36a4ec79c3eb4530627bcbd8c7c549018a19804f1eb43a85c', filename='aclImdb_v1.tar.gz')
To debug, see: https://www.tensorflow.org/datasets/overview#fixing_nonmatchingchecksumerror

In [12]:
dataset = load_dataset("stanfordnlp/imdb")

# Save it to a local folder
dataset.save_to_disk("./imdb_stanfordnlp")
print("Dataset saved to ./imdb_local")

KeyboardInterrupt: 

In [13]:
# Load from local disk (no internet needed)
dataset = load_from_disk("./imdb_stanfordnlp")

train_hf = dataset["train"].shuffle(seed=42).select(range(10000))
test_hf  = dataset["test"].shuffle(seed=42).select(range(5000))

train_data = tf.data.Dataset.from_tensor_slices((
        list(train_hf["text"]),
        list(train_hf["label"])
    ))
test_data  = tf.data.Dataset.from_tensor_slices((
        list(test_hf["text"]),
        list(test_hf["label"])
    ))

print(train_data)
print(test_data)

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>


In [14]:
for review, label in train_data.take(2):
    print("review:")
    print(review.numpy().decode("utf-8")[:700], "...")
    print(label.numpy())
    print("-" * 80)

review:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are real ...
1
--------------------------------------------------------------------------------
review:
This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you st

In [15]:
max_tokens = 10000
sequence_length = 120

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length,
)

train_text_only = train_data.map(lambda text, label: text)

vectorizer.adapt(train_text_only.batch(1024))

vocab = vectorizer.get_vocabulary()

print("Vocabulary size: ", len(vocab))
print("First 20 vocabulary items: ")
print(vocab[:20])
print("-" * 80)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step
Vocabulary size:  10000
First 20 vocabulary items: 
['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('br'), np.str_('was'), np.str_('as'), np.str_('with'), np.str_('for'), np.str_('but'), np.str_('movie')]
--------------------------------------------------------------------------------


In [16]:
for review, label in train_data.take(1):
    one_review = tf.expand_dims(review, axis=0)
    vectorized_review = vectorizer(one_review)

    print("Original review preview:")
    print(review.numpy().decode("utf-8")[:300], "...")

    print("\nVectorized review shape:", vectorized_review.shape)
    print("First 40 token IDs: ")
    print(vectorized_review.numpy()[0][:40])

Original review preview:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot  ...

Vectorized review shape: (1, 120)
First 40 token IDs: 
[  48    7   59 4381   31   32  203    1    3    1   18    2  191   12
  187   24  511  213   43 1081 2769    1  263    1    1  263  353    1
 1801   24  178  581    1  116   24  236   53 2400    1  263]


### Step 12 -- Create Batched Datasets

##### Now we prepare the data for training

###### shuffle()
###### batch()
###### prefetch()

In [17]:
batch_size = 64

train_batches = (
    train_data
    .shuffle(10000, seed=42)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

test_batches = (
    test_data
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

for batch_reviews, batch_labels in train_batches.take(1):
    print("Batch text shape: ", batch_reviews.shape)
    print("Batch label shape: ", batch_labels.shape)
    print("First 10 labels: ", batch_labels[:10].numpy())

Batch text shape:  (64,)
Batch label shape:  (64,)
First 10 labels:  [0 0 0 1 1 0 1 1 1 0]


### Part 3 - Train a Baseline Embedding Model
#### Before we build  the RNN, we train a baseline model:
###### TextVectorization -> Embedding -> GlobalAveragePooling1D -> Dense

In [18]:
embedding_dim = 32

baseline_model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(
        input_dim=max_tokens,
        output_dim=embedding_dim,
        name="word_embedding",
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
baseline_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (1, 120)               │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ word_embedding (Embedding)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
baseline_history = baseline_model.fit(
    train_batches,
    validation_data=test_batches,
    epochs=2,
    shuffle=True,
)

Epoch 1/2


C:\Users\OMID\Documents\PyEnv\PyEnv314\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.6367 - loss: 0.6772 - val_accuracy: 0.6912 - val_loss: 0.6550
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7384 - loss: 0.6152 - val_accuracy: 0.7482 - val_loss: 0.5878


#### Step 15 -- Evaluate The Baseline Model

In [20]:
baseline_loss, baseline_accuracy = baseline_model.evaluate(test_batches, verbose=0)

print(f"Baseline test loss: {baseline_loss:0.4f}")
print(f"Baseline test accuracy: {baseline_accuracy:0.4f}")

Baseline test loss: 0.5878
Baseline test accuracy: 0.7482


## Part 4 - Visualize Word Embedding

### Step 16 -- Extract the Learned Embedding Matrix
##### Now we pull out the learned embedding weights.

In [21]:
embedding_layer = baseline_model.get_layer("word_embedding")
embedding_weights = embedding_layer.get_weights()[0]

print("Embedding matrix shape: ", embedding_weights.shape)
print("Vocabulary size: ", len(vocab))
print("Embedding dimension: ", embedding_weights.shape[1])

Embedding matrix shape:  (10000, 32)
Vocabulary size:  10000
Embedding dimension:  32


### Step 17 - Inspect a Few word vectors

Now we inspect a specific words.

In [30]:
words_to_inspect = ["great", "bad", "excellent", "boring", "movie"]

for word in words_to_inspect:
    if word in vocab:
        word_id = vocab.index(word)
        vector = embedding_weights[word_id]
        print(f"{word:6s} -> ID {word_id:5d} -> first 8 values: {vector[:8]}")
    else:
        print(f"{word:10s} not found in vocabulary")

great  -> ID    86 -> first 8 values: [ 0.41626206 -0.6171259  -0.5485032  -0.456041   -0.541077    0.45520008
 -0.44220012 -0.54348236]
bad    -> ID    83 -> first 8 values: [-0.77886873  0.97263217  0.8489446   0.8110269   0.87737256 -0.7123527
  0.7261422   0.8730406 ]
excellent -> ID   320 -> first 8 values: [ 0.29866314 -0.42394045 -0.32693794 -0.34085715 -0.34112352  0.324278
 -0.32098365 -0.38333657]
boring -> ID   360 -> first 8 values: [-0.33149526  0.45189306  0.40234804  0.41841552  0.39657283 -0.42412865
  0.40425238  0.42851633]
movie  -> ID    19 -> first 8 values: [-0.29724827  0.38430822  0.3092321   0.33020666  0.3321862  -0.2702488
  0.28935593  0.40040454]


### Step 18 - Expert Files for Tensorflow Embedding Projector

The TensorFlow Embedding is an amazing tool that  lets us visualize high-dimentional vectors in 2D or 3D!

In [31]:
num_tokens_to_export = min(2000, len(vocab), embedding_weights.shape[0])

with open("vectors.tsv", "w", encoding="utf-8") as vector_file, open("metadata.tsv", "w", encoding="utf-8") as metadata_file:
    for i in range(1, num_tokens_to_export): # skip index 0, usually padding/empty token
        word = vocab[i]
        vector = embedding_weights[i]
        metadata_file.write(word + "\n")
        vector_file.write("\t".join([str(x) for x in vector]) + "\n")


print("create vectors.tsv and metadata.tsv")
print("Rows exported:", num_tokens_to_export - 1)
print("\nUpload these files to the Tens")
print("https://projector.tensorflow.org/")

create vectors.tsv and metadata.tsv
Rows exported: 1999

Upload these files to the Tens
https://projector.tensorflow.org/


### Step 19 - Why Averaging is Limited

###### The baseline model used GlobalAveragePooling1D.
###### That means it took all the word vectors and averaged them into one sentence vector but averaging has a major limitation!

### Step 20 - What an RNN Does

##### RNN stands for Recurrent Neural Network
###### A simple way to think about it is :
######   An RNN reads a sequence one token at a time while carrying a small memory forward

### Step 21 - Build the SimpleRNN Model

##### Now we build the first RNN classifier!
##### The architecture is:
TextVectorization -> Embedding -> SimpleRNN -> Dense

In [32]:
rnn_model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(
        input_dim=max_tokens,
        output_dim=embedding_dim,
        mask_zero=True,
        name="rnn_embedding",
    ),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
rnn_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (64, 120)              │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Step 22 - Train the RNN Model

###### Now we train the RNN.
###### RNNs can be slower than pooling models because they process sequences step by step
###### That is part of the tradeoff.

In [33]:
rnn_history = rnn_model.fit(
    train_batches,
    validation_data=test_batches,
    epochs=2,
)

Epoch 1/2


C:\Users\OMID\Documents\PyEnv\PyEnv314\Lib\site-packages\keras\src\layers\layer.py:1092: UserWarning: Layer 'simple_rnn' (of type SimpleRNN) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.5082 - loss: 0.6936 - val_accuracy: 0.5348 - val_loss: 0.6902
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.7160 - loss: 0.5977 - val_accuracy: 0.6690 - val_loss: 0.6303


### Step 23 - Evaluate the RNN Model

###### Now we evaluate the RNN on the test data.
###### This gives us a test loss and test accuracy!

In [34]:
rnn_loss, rnn_accuracy = rnn_model.evaluate(test_batches, verbose=0)

print(f"RNN test loss: {rnn_loss:0.4f}")
print(f"RNN test accuracy: {rnn_accuracy:0.4f}")

RNN test loss: 0.6303
RNN test accuracy: 0.6690


### Step 24 - Compare the Baseline and RNN

###### Now we print both model results together!

In [35]:
print("Baseline Model")
print(f"Loss: {baseline_loss:0.4f} | Accuracy: {baseline_accuracy:0.4f}")

print("\n RNN Model")
print(f"Loss: {rnn_loss:0.4f} | Accuracy: {rnn_accuracy:0.4f}")

Baseline Model
Loss: 0.5878 | Accuracy: 0.7482

 RNN Model
Loss: 0.6303 | Accuracy: 0.6690
